# Migrate workspace → `pizer.egr.duke.edu`

Copies `/data/users/kv133/Alpha-Earth-Embeddings-Research` **as-is** to
`kv133@pizer.egr.duke.edu` with:

- password via `getpass` (not saved to disk)
- resumable `rsync`
- `tqdm` byte + file progress

Helper: `scripts/migrate_to_pizer.py`. Re-run the transfer cell to resume.

In [25]:
from __future__ import annotations

import getpass
import importlib
import sys
from pathlib import Path

REPO = Path("/data/users/kv133/Alpha-Earth-Embeddings-Research").resolve()
sys.path.insert(0, str(REPO / "scripts"))

import migrate_to_pizer as _mig

_mig = importlib.reload(_mig)  # pick up script edits without kernel restart
MigrateConfig = _mig.MigrateConfig
migrate = _mig.migrate
local_size_bytes = _mig.local_size_bytes

assert "bash -lc" in open(_mig.__file__).read()
print("REPO =", REPO)
print("helper =", _mig.__file__)
print("exists =", REPO.is_dir())

REPO = /data/users/kv133/Alpha-Earth-Embeddings-Research
helper = /data/users/kv133/Alpha-Earth-Embeddings-Research/scripts/migrate_to_pizer.py
exists = True


In [26]:
# ── edit if needed ──────────────────────────────────────────────────────
REMOTE_USER = "kv133"
REMOTE_HOST = "pizer.egr.duke.edu"

# None = auto-pick /data/users/... if writable, else ~/
REMOTE_DEST = None

# Empty = true as-is. Example skips:
# EXCLUDES = [".venv/", "__pycache__/", ".ipynb_checkpoints/"]
EXCLUDES: list[str] = []

DRY_RUN = False  # True = rsync dry-run only

cfg = MigrateConfig(
    src=REPO,
    remote_user=REMOTE_USER,
    remote_host=REMOTE_HOST,
    remote_dest=REMOTE_DEST,
    excludes=tuple(EXCLUDES),
    dry_run=DRY_RUN,
)
print(cfg)

MigrateConfig(src=PosixPath('/data/users/kv133/Alpha-Earth-Embeddings-Research'), remote_user='kv133', remote_host='pizer.egr.duke.edu', remote_port=22, remote_dest=None, excludes=(), dry_run=False, password=None)


In [27]:
# Type password here (hidden). Lives only in this kernel.
PASSWORD = getpass.getpass(f"Password for {REMOTE_USER}@{REMOTE_HOST}: ")
assert PASSWORD, "empty password"
cfg.password = PASSWORD
print("Password captured (not printed).")

Password captured (not printed).


In [28]:
# Optional: local size estimate (~1 min on ~34G)
approx = local_size_bytes(REPO, EXCLUDES)
print(f"approx {approx:,} bytes ({approx / 1024**3:.2f} GiB)")

approx 35,648,889,344 bytes (33.20 GiB)


In [ ]:
# Transfer (safe to re-run — rsync resumes via --partial)
result = migrate(cfg)
result

local src: /data/users/kv133/Alpha-Earth-Embeddings-Research
probing remote …


remote dest: kv133@pizer.egr.duke.edu:/data/users/kv133/Alpha-Earth-Embeddings-Research
 
pizer.egr.duke.edu
Filesystem      Size  Used Avail Use% Mounted on
/dev/sda1       3.6T  2.8T  710G  80% /data
/dev/sda1       3.6T  2.8T  710G  80% /data
1.5G	/data/users/kv133/Alpha-Earth-Embeddings-Research

estimating local size (may take a minute) …
approx local bytes: 35,648,889,344 (33.20 GiB)
opening SSH control master (one password prompt) …
starting rsync (incremental — files should move within seconds; bar may jump as totals refine) …


rsync → pizer:   0%|          | 0.00/33.2G [00:00<?, ?B/s]

files xfr'd: 0file [00:00, ?file/s]

### After it finishes on pizer

```bash
ssh kv133@pizer.egr.duke.edu
cd /data/users/kv133/Alpha-Earth-Embeddings-Research   # or the path printed above
# recreate venv if binary-incompatible:
# python3 -m venv .venv && . .venv/bin/activate && pip install -r requirements.txt
```

`pgdata/` was copied as-is; start Postgres only if you need the international DB on pizer.